# Fraud screening model

Heavily imbalanced target (~10% fraud) — class balance is reported and AUC accompanies accuracy.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

In [2]:
rng = np.random.default_rng(23)
n = 3000
df = pd.DataFrame({
    "amount": rng.gamma(2, 40, n).round(2),
    "merchant_risk": rng.beta(2, 5, n).round(3),
    "hour_of_day": rng.integers(0, 24, n),
    "card_age_days": rng.gamma(4, 200, n).round(0),
    "tx_last_24h": rng.poisson(2.2, n),
    "distance_from_home": rng.gamma(1.5, 30, n).round(1),
    "failed_auths": rng.poisson(0.3, n),
    "new_device": rng.integers(0, 2, n),
    "amount_vs_avg": rng.normal(1, 0.6, n).round(2),
    "country_risk": rng.beta(1.5, 8, n).round(3),
})
score = (0.004 * df["amount"] + 0.8 * df["merchant_risk"]
         + 0.5 * df["failed_auths"] + rng.normal(0, 1.6, n))
df["fraud"] = (score > np.quantile(score, 0.9)).astype(int)

In [3]:
df["fraud"].value_counts(normalize=True)

fraud
0    0.9
1    0.1
Name: proportion, dtype: float64

In [4]:
num_cols = ['amount', 'merchant_risk', 'hour_of_day', 'card_age_days', 'tx_last_24h', 'distance_from_home', 'failed_auths', 'new_device', 'amount_vs_avg', 'country_risk']
X = df[num_cols]
y = df["fraud"]

In [5]:
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=23, stratify=y)
scaler = StandardScaler()
X_tr = scaler.fit_transform(X_tr)
X_te = scaler.transform(X_te)

In [ ]:
model = LogisticRegression(max_iter=1000)
model.fit(X_tr, y_tr)
pred = model.predict(X_te)
acc = accuracy_score(y_te, pred)

The classifier is highly accurate on held-out data — performance is strong and the model is ready to use.